In [5]:
import pandas as pd
w = pd.read_json("../../experiment_runs/main_framework_run/2wikimultihopqa/system1/parsed_responses.jsonl", lines=True)
h = pd.read_json("../../experiment_runs/main_framework_run/hotpotqa/system1/parsed_responses.jsonl", lines=True)
m = pd.read_json("../../experiment_runs/main_framework_run/musique/system1/parsed_responses.jsonl", lines=True)
main = pd.concat([w,h,m])

In [6]:
main

,question,answer,generation,avg_log_prob,system_1_guess
0,Where was the place of burial of Charles Mathe...,Cork,system\nYou are a precise answering engine. Yo...,0.030670,Westminster Abbey
1,"Which film has the director who died later, Th...","Sleep, My Love",system\nYou are a precise answering engine. Yo...,0.104117,"Sleep, My Love"
2,Where was the husband of Octavie Coudreau born?,Sonnac,system\nYou are a precise answering engine. Yo...,0.004315,France
3,"Which film has the director who died earlier, ...",Heinz In The Moon,system\nYou are a precise answering engine. Yo...,0.254024,Heinz In The Moon
4,What is the place of birth of Queen Jeongsun (...,Korea,system\nYou are a precise answering engine. Yo...,0.030696,Jeolla Province
...,...,...,...,...,...
495,What year did the author of History of the Ref...,1572,system\nYou are a precise answering engine. Yo...,0.081771,1580
496,Who was a prominent figure at the radio divisi...,Walter Sabo,system\nYou are a precise answering engine. Yo...,0.008964,ABC Network
497,Where did the founder of the Acting Company st...,Clifton College,system\nYou are a precise answering engine. Yo...,0.247405,Yale School of Drama
498,What is the mascot of the operator of RV Wecoma?,Benny Beaver,system\nYou are a precise answering engine. Yo...,0.007821,Torii


In [2]:
from statsmodels.stats.contingency_tables import mcnemar
from evaluate import load
from tqdm import tqdm
import pandas as pd

# standard-RAG
w = pd.read_json("../../experiment_runs/naive/2wikimultihopqa_final_responses.jsonl", lines=True)
h = pd.read_json("../../experiment_runs/naive/hotpotqa_final_responses.jsonl", lines=True)
m = pd.read_json("../../experiment_runs/naive/musique_final_responses.jsonl", lines=True)
naive_rag = pd.concat([w, h, m])
naive_rag.rename(columns={'answer': 'gold', 'final_ans': 'naive_rag_answer'}, inplace=True)

# framework
w = pd.read_json("../../experiment_runs/main_framework_run/2wikimultihopqa/system2/final_response/final_responses.jsonl", lines=True)
h = pd.read_json("../../experiment_runs/main_framework_run/hotpotqa/system2/final_response/final_responses.jsonl", lines=True)
m = pd.read_json("../../experiment_runs/main_framework_run/musique/system2/final_response/final_responses.jsonl", lines=True)
main = pd.concat([w,h,m])
main.drop(columns=['system_1_guess'], inplace=True)
main.rename(columns={'answer': 'gold', 'final_ans': 'main_answer'}, inplace=True)

combined_df = pd.merge(main, naive_rag)
squad_metric = load("squad_v2")

# a = both correct
# b = main correct, baseline wrong
# c = main wrong, baseline correct
# d = both wrong

a, b, c, d = 0, 0, 0, 0

for row in tqdm(combined_df.itertuples()):
    naive_pred, main_pred, reference = [], [], []
    main_answer, naive_rag_answer, gold = row.main_answer, row.naive_rag_answer, row.gold
    reference.append({'answers': {'answer_start': [0], 'text': [gold]}, 'id': '0'})
    
    # Check naive
    naive_pred.append({'prediction_text': naive_rag_answer, 'id': "0", 'no_answer_probability': 0.})
    naive_em = squad_metric.compute(predictions=naive_pred, references=reference)['exact']
    
    # Check main
    main_pred.append({'prediction_text': main_answer, 'id': "0", 'no_answer_probability': 0.})
    main_em = squad_metric.compute(predictions=main_pred, references=reference)['exact']

    if main_em == 100.0 and naive_em == 100.0:
        a += 1
    elif main_em == 100.0 and naive_em != 100.0:
        b += 1
    elif main_em != 100.0 and naive_em == 100.0:
        c += 1
    else:
        d += 1

table = [[a, b],
         [c, d]]

result = mcnemar(table, correction=True)
print(f"McNemar's test p-value: {result.pvalue}")

1500it [00:26, 56.84it/s]

McNemar's test p-value: 0.012313072292256703


In [3]:
a,b,c,d

(133, 170, 126, 1071)